# ETL y EDA — 03_hu_2022_pm1_salud
Cuaderno reproducible para extraer texto del PDF original, normalizarlo, perfilar términos y visualizar la cobertura de la evidencia curada.

**Referencia APA 7:** Hu, Y., Wu, M., Li, Y., & Liu, X. (2022). Influence of PM1 exposure on total and cause-specific respiratory diseases: A systematic review and meta-analysis. Environmental Science and Pollution Research, 29, 15117–15126. https://doi.org/10.1007/s11356-021-16536-0


In [ ]:
from pathlib import Path
import csv, json, re
from collections import Counter
from PIL import Image, ImageDraw, ImageFont
from pypdf import PdfReader

SOURCE = json.loads(r'''{
  "id": "03_hu_2022_pm1_salud",
  "pdf": "fuentes_academicas/03_hu_2022_pm1_salud/Hu_et_al_2022_PM1.pdf",
  "apa": "Hu, Y., Wu, M., Li, Y., & Liu, X. (2022). Influence of PM1 exposure on total and cause-specific respiratory diseases: A systematic review and meta-analysis. Environmental Science and Pollution Research, 29, 15117–15126. https://doi.org/10.1007/s11356-021-16536-0",
  "url": "https://doi.org/10.1007/s11356-021-16536-0",
  "terms": [
    "PM1",
    "respiratory",
    "asthma",
    "pneumonia",
    "heterogeneity",
    "publication bias"
  ]
}''')
cwd = Path.cwd()
root = cwd.parents[1] if cwd.name == SOURCE["id"] else cwd
pdf_path = root / SOURCE["pdf"]
out = root / "extraccion_academica" / SOURCE["id"]
reader = PdfReader(str(pdf_path))
raw = "\n\n".join(page.extract_text() or "" for page in reader.pages)
clean = raw.replace("\u00ad", "").replace("\u00a0", " ")
clean = re.sub(r"(?<=\w)-\n(?=\w)", "", clean)
clean = re.sub(r"[ \t]+", " ", clean)
clean = re.sub(r"\n{3,}", "\n\n", clean).strip()
(out / "texto_normalizado.txt").write_text(clean, encoding="utf-8")
counts = [{"termino": term, "apariciones": len(re.findall(re.escape(term), clean, re.I))} for term in SOURCE["terms"]]
with (out / "perfil_terminos.csv").open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["termino", "apariciones"])
    writer.writeheader(); writer.writerows(counts)
counts

## EDA de la evidencia curada
El gráfico resume dimensiones, no magnitudes de riesgo ni causalidad individual.


In [ ]:
evidence = json.loads((out / "evidencia_curada.json").read_text(encoding="utf-8"))
dimensions = Counter(row["dimension"] for row in evidence)
image = Image.new("RGB", (1100, 620), "#f5f1e8")
draw = ImageDraw.Draw(image)
font = ImageFont.load_default(size=22)
draw.text((70, 48), "Cobertura de evidencia por dimensión", fill="#173d3a", font=ImageFont.load_default(size=34))
items = list(dimensions.items()); maximum = max(dimensions.values(), default=1)
for index, (label, value) in enumerate(items):
    slot = 900 / max(len(items), 1); left = 100 + index * slot; bar_width = min(120, slot * .56)
    bar_height = 310 * value / maximum
    draw.rounded_rectangle((left, 510 - bar_height, left + bar_width, 510), radius=12, fill="#2f736a")
    draw.text((left, 528), label, fill="#33494a", font=font)
image.save(out / "cobertura_evidencia.png")
image